In [ ]:
#module1
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 62.1/112.6 GB disk)


In [ ]:
!pip install -q pypdf sentence-transformers
# nếu bạn dùng FAISS:
!pip install -q faiss-cpu  || echo "FAISS install failed → sẽ dùng numpy fallback"

In [ ]:
#copy link bên roboflow

In [ ]:
# Train YOLO11n on COCO8 for 3 epochs
!yolo train model=yolo11n.pt data=/content/My-Traffic-Train-3/data.yaml epochs=100 imgsz=640

In [ ]:
# ================== MODULE 1 (YOLOv11): Keyframe -> Detect -> JSON/CSV ==================
# ==== 0) IMPORTS ====
import os, re, json, csv, gc, time
from pathlib import Path
from typing import List, Dict, Any, Tuple
import numpy as np
import cv2
from ultralytics import YOLO
try:
    import torch
except Exception:
    torch = None

# ==== 1) KHAI BÁO ĐƯỜNG DẪN & THAM SỐ (SỬA Ở ĐÂY) ====
WEIGHTS            = "/content/data/best.pt"       # hoặc "yolo11s.pt"
VIDEO_IN           = "/content/drive/MyDrive/videos"              # file .mp4 hoặc folder video
OUT_DIR            = "/content/runs_meta"
META_JSON          = f"{OUT_DIR}/metadata.json"
DET_CSV            = f"{OUT_DIR}/detections.csv"

# Keyframe
KEYFRAME_MODE      = "time"   # "time" hoặc "scene"
SECONDS_PER_SAMPLE = 1.0
MAX_FRAMES_PER_VID = 60       # 0 = không giới hạn
SCENE_THRESHOLD    = 0.40

# YOLO
CONF_THRES         = 0.25
IOU_THRES          = 0.45
if (torch and torch.cuda.is_available()) or os.path.exists("/proc/driver/nvidia/version"):
    DEVICE = 0
else:
    DEVICE = "cpu"
BATCH              = 8

# ==== 2) TIỆN ÍCH ====
def ensure_dir(p: str | Path): Path(p).mkdir(parents=True, exist_ok=True)
def norm_vpath(p: str) -> str:
    if not p: return ""
    return re.sub(r"/{2,}", "/", p.replace("\\", "/").strip())

def get_video_info(vid_path: str | Path) -> Tuple[float, float, int]:
    cap = cv2.VideoCapture(str(vid_path))
    if not cap.isOpened(): return 0.0, 0.0, 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = frames / fps if fps > 0 else 0.0
    cap.release()
    return fps, duration, frames

def list_videos(input_path: str | Path) -> List[Path]:
    p = Path(input_path)
    if p.is_file(): return [p]
    exts = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}
    return sorted([q for q in p.rglob("*") if q.suffix.lower() in exts])

# ==== 3) KEYFRAME ====
def sample_timestamps_by_time(duration: float, step: float, limit: int) -> List[float]:
    if duration <= 0: return []
    ts = [round(t, 3) for t in np.arange(0.0, duration, step).tolist()]
    return ts[:limit] if limit > 0 else ts

def sample_timestamps_by_scene(vid_path: str | Path, thr: float, limit: int) -> List[float]:
    cap = cv2.VideoCapture(str(vid_path))
    if not cap.isOpened(): return []
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    if fps <= 0: cap.release(); return []
    prev_hist, timestamps, idx = None, [0.0], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hist = cv2.normalize(cv2.calcHist([gray],[0],None,[64],[0,256]), None).flatten()
        if prev_hist is not None:
            diff = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_BHATTACHARYYA)  # 0~1
            if diff >= thr:
                t = idx / fps
                if not timestamps or (t - timestamps[-1] >= 0.25):
                    timestamps.append(round(t, 3))
                    if limit > 0 and len(timestamps) >= limit: break
        prev_hist, idx = hist, idx + 1
    cap.release()
    return timestamps

def read_frame_at(vid_path: str | Path, t_sec: float):
    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    if fps <= 0: cap.release(); return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(t_sec * fps))
    ok, frame = cap.read(); cap.release()
    return frame if ok else None

def extract_keyframes(vid_path: Path) -> Tuple[List[float], List[np.ndarray]]:
    _, duration, _ = get_video_info(vid_path)
    if KEYFRAME_MODE == "scene":
        ts = sample_timestamps_by_scene(vid_path, SCENE_THRESHOLD, MAX_FRAMES_PER_VID)
    else:
        ts = sample_timestamps_by_time(duration, SECONDS_PER_SAMPLE, MAX_FRAMES_PER_VID)
    imgs = [read_frame_at(vid_path, t) for t in ts]
    return ts, imgs

# ==== 4) YOLOv11 INFERENCE ====
def auto_find_best_weights(root="runs/detect"):
    rootp = Path(root)
    if not rootp.exists(): return None
    cands = list(rootp.glob("**/weights/best.pt"))
    if not cands: return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(cands[0])

def init_model(weights: str | None) -> YOLO:
    if not weights:
        w = auto_find_best_weights()
        weights = w if w else "yolo11s.pt"
        print(f"[INFO] dùng weights: {weights}")
    else:
        if Path(weights).suffix.lower() == ".pt":
            assert Path(weights).exists(), f"Không thấy weights: {weights}"
    print(f"[YOLOv11] Loading: {weights}")
    return YOLO(weights)

def run_yolo_on_frames(model: YOLO, frames: List[np.ndarray], conf=0.25, iou=0.45, device="cpu", batch=8):
    results = [None] * len(frames)
    idxs = [i for i,f in enumerate(frames) if f is not None]
    if not idxs: return results
    for k in range(0, len(idxs), batch):
        bi = idxs[k:k+batch]
        imgs = [frames[i] for i in bi]
        r = model.predict(source=imgs, conf=conf, iou=iou, device=device, verbose=False)
        for j, ridx in enumerate(bi): results[ridx] = r[j]
    return results

def postprocess_results(result, names: Dict[int,str]) -> List[Dict[str,Any]]:
    if result is None or not hasattr(result, "boxes"): return []
    boxes = result.boxes
    if boxes is None: return []
    xyxy = boxes.xyxy.cpu().numpy() if hasattr(boxes, "xyxy") else []
    cls  = boxes.cls.cpu().numpy().astype(int) if hasattr(boxes, "cls") else []
    conf = boxes.conf.cpu().numpy() if hasattr(boxes, "conf") else []
    out = []
    for i in range(len(cls)):
        x1,y1,x2,y2 = map(float, xyxy[i]); c = int(cls[i])
        out.append({"cls": c, "name": names.get(c, str(c)), "conf": float(conf[i]), "xyxy": [x1,y1,x2,y2]})
    return out

# ==== 5) PIPELINE & CHẠY ====
def pipeline_module1_to_files(videos: List[Path], weights: str | None, meta_json: str, det_csv: str):
    ensure_dir(Path(meta_json).parent)
    model = init_model(weights)
    names = getattr(model.model, "names", None) or getattr(model, "names", {}) or {}

    all_meta = []
    with open(det_csv, "w", encoding="utf-8", newline="") as fcsv:
        wr = csv.DictWriter(fcsv, fieldnames=["video_path","t","cls","name","conf","x1","y1","x2","y2"])
        wr.writeheader()

        for i, vp in enumerate(list_videos(VIDEO_IN), 1):
            vpath = norm_vpath(str(vp))
            fps, duration, frames = get_video_info(vp)
            print(f"[{i}] {vp.name} | fps={fps:.2f} dur={duration:.1f}s frames={frames}")

            ts, imgs = extract_keyframes(vp)
            print(f"  - keyframes: {len(ts)}")

            yres = run_yolo_on_frames(model, imgs, conf=CONF_THRES, iou=IOU_THRES, device=DEVICE, batch=BATCH)

            kf = []
            for j, t in enumerate(ts):
                objs = postprocess_results(yres[j], names)
                kf.append({"t": float(t), "objects": objs})
                for o in objs:
                    wr.writerow({
                        "video_path": vpath, "t": float(t), "cls": o["cls"], "name": o["name"], "conf": o["conf"],
                        "x1": o["xyxy"][0], "y1": o["xyxy"][1], "x2": o["xyxy"][2], "y2": o["xyxy"][3],
                    })

            all_meta.append({
                "video_path": vpath, "fps": float(fps), "duration": float(duration),
                "frames": int(frames), "keyframes": kf
            })

            del imgs, yres, kf
            gc.collect()

    with open(meta_json, "w", encoding="utf-8") as f:
        json.dump({"created_at": time.time(), "video_count": len(all_meta), "items": all_meta},
                  f, ensure_ascii=False, indent=2)
    print(f"[OK] Saved metadata:   {meta_json}")
    print(f"[OK] Saved detections: {det_csv}")

# ==== RUN ====
assert Path(VIDEO_IN).exists(), f"Không thấy VIDEO_IN: {VIDEO_IN}"
ensure_dir(OUT_DIR)
pipeline_module1_to_files(list_videos(VIDEO_IN), WEIGHTS, META_JSON, DET_CSV)

# Preview nhanh
try:
    data = json.load(open(META_JSON, "r", encoding="utf-8"))
    print("Preview item 1:\n", json.dumps(data["items"][0], ensure_ascii=False, indent=2)[:1000])
except Exception as e:
    print("Preview lỗi:", e)


[YOLOv11] Loading: /content/data/best.pt
[1] 01f7c7ec_1000_clip_006_0040_0050_Y.mp4 | fps=30.00 dur=10.0s frames=300
  - keyframes: 10
[2] 0577aea9_865_clip_021_0161_0166_Y.mp4 | fps=30.00 dur=4.2s frames=125
  - keyframes: 0
[3] 0758e201_800_clip_014_0102_0109_Y.mp4 | fps=30.00 dur=5.8s frames=175
  - keyframes: 6
[4] 08e895d7_985_clip_003_0010_0017_N.mp4 | fps=30.00 dur=7.0s frames=210
  - keyframes: 7
[5] 0d77c5a0_792_clip_008_0052_0060_N.mp4 | fps=30.00 dur=8.0s frames=240
  - keyframes: 8
[6] 0f5675f8_801_clip_008_0050_0060_Y.mp4 | fps=30.00 dur=10.0s frames=300
  - keyframes: 10
[7] 113634a5_900_clip_008_0044_0054_Y.mp4 | fps=30.00 dur=8.3s frames=250
  - keyframes: 9
[8] 12d2fa06_1000_clip_008_0054_0060_Y.mp4 | fps=30.00 dur=6.0s frames=180
  - keyframes: 6
[9] 175ba98d_1000_clip_002_0008_0018_Y.mp4 | fps=30.00 dur=10.0s frames=300
  - keyframes: 10
[10] 17be3111_792_clip_005_0027_0037_N.mp4 | fps=30.00 dur=10.0s frames=300
  - keyframes: 10
[11] 17d2582e_792_clip_009_0051_0060_

KeyboardInterrupt: 

Module 2


In [ ]:
# ==== MODULE 2 - COMMON SETUP ====
# Chỉnh ở đây để khớp dự án của bạn
RB_ROOT   = "roadbuddy"
FRAMES_DIR= f"{RB_ROOT}/data/frames"
RUNTIME_DIR=f"{RB_ROOT}/runtime"
KB_DIR    = f"{RB_ROOT}/kb"
PDF_DIR   = f"{KB_DIR}/pdfs"

SCENE_CACHE_PATH = f"{RUNTIME_DIR}/scene_cache.jsonl"        # từ bước detect (YOLO) đã chạy
SCENE_DESC_PATH  = f"{RUNTIME_DIR}/scene_description.jsonl"  # output M2.1

# VLM & Instruct settings
VLM_MODEL = "Salesforce/instructblip-vicuna-7b"  # có thể đổi sang flan-t5-xl nếu GPU yếu
FPS       = 15    # FPS tương ứng khi bạn trích frames
WIN_SEC   = 3     # độ dài cửa sổ mô tả cảnh
CONF_TH   = 0.35  # ngưỡng confidence

# Class names (đổi theo training của bạn)
SIGN_NAMES  = ["speed_limit","no_left","no_right","stop","priority","give_way","end_speed","other_sign"]
LIGHT_NAMES = ["red","yellow","green","green_left","green_straight","red_yellow","off"]

# Embedding model cho RAG
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RAG_OUT   = f"{KB_DIR}/index"
CHUNKS_JSONL = f"{RAG_OUT}/chunks.jsonl"
FAISS_INDEX  = f"{RAG_OUT}/faiss.index"

# ===== Imports =====
import os, json, re, gzip, glob, math, shutil
from pathlib import Path
from collections import defaultdict, Counter
from typing import List, Dict, Any, Iterable


In [ ]:
# ==== Convert detections.csv -> scene_cache.jsonl, rồi chạy Scene Description ====
import os, json, math
from pathlib import Path
import pandas as pd

CSV_IN      = "/content/runs_meta/detections.csv"   # <-- file của bạn
RB_ROOT     = "roadbuddy"
OUT_JSONL   = f"{RB_ROOT}/runtime/scene_cache.jsonl"
SCENE_DESC  = f"{RB_ROOT}/runtime/scene_description.jsonl"

# Fallback FPS nếu CSV không có cột fps
FPS_FALLBACK = 30.0  # (ảnh bạn gửi có fps=30)

# Heuristic tách đèn vs biển (điều chỉnh theo nhãn thực tế)
LIGHT_NAME_HINTS = {"den", "denDo", "denXanh", "denVang", "den_muiTen", "traffic_light"}

def is_light(name: str) -> bool:
    n = (str(name) or "").lower()
    return any(h.lower() in n for h in LIGHT_NAME_HINTS)

# ---- Đọc CSV & chuẩn hoá ----
df = pd.read_csv(CSV_IN)

# Chuẩn tên cột tối thiểu
required_any = {"video_path", "name", "cls", "conf"}
if not required_any.issubset(df.columns):
    raise RuntimeError(f"CSV thiếu cột bắt buộc: {required_any - set(df.columns)}")

# Hỗ trợ cả 2 trường hợp: có 't' (giây) hoặc 'frame'
has_t     = "t"     in df.columns
has_frame = "frame" in df.columns
has_fps   = "fps"   in df.columns

if not (has_t or has_frame):
    raise RuntimeError("CSV cần có cột 't' (giây) hoặc 'frame' (số khung).")

# Điền FPS nếu thiếu
if not has_fps:
    df["fps"] = FPS_FALLBACK

# Nếu chưa có 'frame' -> tính từ t * fps; nếu chưa có 't' -> tính từ frame / fps
if not has_frame:
    df["frame"] = (df["t"] * df["fps"]).round().astype(int)
if not has_t:
    df["t"] = df["frame"] / df["fps"]

# Bbox: hỗ trợ x1,y1,x2,y2 hoặc xyxy(list-as-string)
if {"x1","y1","x2","y2"}.issubset(df.columns):
    pass
elif "xyxy" in df.columns:
    # tách chuỗi "[x1, y1, x2, y2]"
    def _parse_xy(s):
        try:
            s = str(s).replace("[","").replace("]","").replace(","," ")
            xs = [float(v) for v in s.split()]
            return xs if len(xs)==4 else [0,0,0,0]
        except: return [0,0,0,0]
    tmp = df["xyxy"].apply(_parse_xy)
    df["x1"] = tmp.apply(lambda v: v[0])
    df["y1"] = tmp.apply(lambda v: v[1])
    df["x2"] = tmp.apply(lambda v: v[2])
    df["y2"] = tmp.apply(lambda v: v[3])
else:
    # không có bbox thì tạo rỗng
    for c in ["x1","y1","x2","y2"]:
        df[c] = 0.0

# ---- Gom theo (video_path, frame) -> ghi JSONL chuẩn scene_cache ----
Path(f"{RB_ROOT}/runtime").mkdir(parents=True, exist_ok=True)

count = 0
with open(OUT_JSONL, "w", encoding="utf-8") as out:
    for (vpath, frame), g in df.groupby(["video_path", "frame"]):
        vid = os.path.splitext(os.path.basename(str(vpath)))[0] or "vid"
        rec = {"vid": vid, "frame": int(frame), "signs": [], "lights": [], "lanes": [], "ocr": []}
        for r in g.itertuples(index=False):
            entry = {
                "cls":  int(getattr(r, "cls")),
                "conf": float(getattr(r, "conf")),
                "bbox": [float(getattr(r,"x1",0.0)), float(getattr(r,"y1",0.0)),
                         float(getattr(r,"x2",0.0)), float(getattr(r,"y2",0.0))]
            }
            name = getattr(r, "name", "")
            (rec["lights"] if is_light(name) else rec["signs"]).append(entry)
        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        count += 1

print(f"✅ Wrote {count} frames -> {OUT_JSONL}")

# ---- Gọi Scene Description nếu đã có hàm build_scene_descriptions() trong notebook ----
try:
    build_scene_descriptions(scene_cache_path=OUT_JSONL, out_path=SCENE_DESC)
    # preview
    import itertools
    print("── Preview scene_description.jsonl ──")
    with open(SCENE_DESC, "r", encoding="utf-8") as f:
        print("".join(list(itertools.islice(f, 5))) or "(trống)")
except NameError:
    print("ℹ️ Chưa có hàm build_scene_descriptions() trong notebook. Chạy cell 2.1 trước rồi gọi lại.")


✅ Wrote 651 frames -> roadbuddy/runtime/scene_cache.jsonl
✅ Scene Description: 151 đoạn → roadbuddy/runtime/scene_description.jsonl
── Preview scene_description.jsonl ──
{"vid": "01f7c7ec_1000_clip_006_0040_0050_Y", "t0": 0.0, "t1": 18.0, "description": "Biển: sign_45×4, sign_175×2, sign_295×1."}
{"vid": "0577aea9_865_clip_021_0161_0166_Y", "t0": 4.0, "t1": 8.0, "description": "Biển: sign_45×2."}
{"vid": "0758e201_800_clip_014_0102_0109_Y", "t0": 2.0, "t1": 6.0, "description": "Biển: sign_46×1."}
{"vid": "08e895d7_985_clip_003_0010_0017_N", "t0": 0.0, "t1": 12.0, "description": "Biển: sign_175×2, sign_45×1."}
{"vid": "0d77c5a0_792_clip_008_0052_0060_N", "t0": 2.0, "t1": 12.0, "description": "Biển: sign_293×2."}



In [ ]:
# ===================== Cell 2.1 — Scene Description =====================

def _open_any(path: str):
    import gzip
    return gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, "r", encoding="utf-8")

def _load_scene_cache(path: str) -> Iterable[Dict[str, Any]]:
    with _open_any(path) as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception as e:
                print(f"[WARN] Bỏ dòng {ln}: JSON lỗi ({e})"); continue
            if "vid" not in rec or "frame" not in rec:
                print(f"[WARN] Bỏ dòng {ln}: thiếu 'vid' hoặc 'frame'"); continue
            rec.setdefault("signs", [])
            rec.setdefault("lights", [])
            rec.setdefault("lanes", [])
            rec.setdefault("ocr", [])
            yield rec

def _label_name(names: List[str], idx: int, prefix: str) -> str:
    return names[idx] if 0 <= idx < len(names) else f"{prefix}_{idx}"

def _summarize_window(records: List[Dict[str, Any]], conf_thres: float) -> str:
    signs, lights, lane_flags, ocr_vals = Counter(), Counter(), Counter(), Counter()
    for r in records:
        for s in r.get("signs", []):
            try:
                if float(s.get("conf", 0)) >= conf_thres:
                    cname = _label_name(SIGN_NAMES, int(s.get("cls", -1)), "sign")
                    signs[cname] += 1
            except: pass
        for l in r.get("lights", []):
            try:
                if float(l.get("conf", 0)) >= conf_thres:
                    cname = _label_name(LIGHT_NAMES, int(l.get("cls", -1)), "light")
                    lights[cname] += 1
            except: pass
        for ln in r.get("lanes", []):
            t = ln.get("type")
            if t: lane_flags[t] += 1
        for o in r.get("ocr", []):
            txt = str(o.get("text", "")).strip()
            if txt: ocr_vals[txt] += 1

    parts = []
    if signs:      parts.append("Biển: " + ", ".join(f"{k}×{v}" for k,v in signs.most_common(5)) + ".")
    if lights:     parts.append("Đèn: " + ", ".join(f"{k}×{v}" for k,v in lights.most_common(5)) + ".")
    if lane_flags: parts.append("Làn: " + ", ".join(f"{k}×{v}" for k,v in lane_flags.most_common(4)) + ".")
    if ocr_vals:   parts.append("OCR: " + ", ".join(f"{k}×{v}" for k,v in ocr_vals.most_common(4)) + ".")
    if not parts:  parts.append("Không có đặc trưng nổi bật trong cửa sổ.")
    return " ".join(parts)

def build_scene_descriptions(scene_cache_path=SCENE_CACHE_PATH,
                             out_path=SCENE_DESC_PATH,
                             fps=FPS, win_sec=WIN_SEC, conf_th=CONF_TH):
    if not os.path.isfile(scene_cache_path):
        raise FileNotFoundError(
            f"Không thấy scene_cache: {scene_cache_path}\n"
            f"→ Tạo từ detections.csv hoặc chỉnh SCENE_CACHE_PATH cho đúng."
        )
    Path(os.path.dirname(out_path) or ".").mkdir(parents=True, exist_ok=True)

    by_vid = defaultdict(list)
    total = 0
    for rec in _load_scene_cache(scene_cache_path):
        by_vid[rec["vid"]].append(rec); total += 1
    if total == 0:
        raise RuntimeError("scene_cache rỗng (0 dòng).")

    for vid in by_vid:
        by_vid[vid].sort(key=lambda r: int(r["frame"]))

    win_frames = max(1, int(fps * win_sec))
    wrote = 0
    with open(out_path, "w", encoding="utf-8") as out:
        for vid, recs in by_vid.items():
            i, n = 0, len(recs)
            while i < n:
                j = min(i + win_frames, n)
                window = recs[i:j]
                t0 = round(int(window[0]["frame"]) / float(fps), 3)
                t1 = round(int(window[-1]["frame"]) / float(fps), 3)
                desc = _summarize_window(window, conf_th)
                out.write(json.dumps({"vid": vid, "t0": t0, "t1": t1, "description": desc}, ensure_ascii=False) + "\n")
                wrote += 1
                i = j
    print(f"✅ Scene Description: {wrote} đoạn → {out_path}")

# chạy ngay:
build_scene_descriptions()


✅ Scene Description: 151 đoạn → roadbuddy/runtime/scene_description.jsonl


In [ ]:
# ==== M2.2: VLM Instruction ====
VLM_SYSTEM_PROMPT = (
  "Bạn là trợ lý lái xe RoadBuddy. Trả lời ngắn gọn, chắc chắn, ưu tiên an toàn "
  "và tuân thủ Luật Giao thông đường bộ Việt Nam. Khi đèn/biển/vạch xung đột, "
  "ưu tiên đèn > biển > vạch. Khi không đủ thông tin, hãy nói 'chưa đủ thông tin' "
  "và khuyên 'giảm tốc/giữ làn'. Tránh suy đoán."
)

def build_user_prompt(question: str, scene_text: str = None):
    ctx = f"Cảnh tóm tắt: {scene_text}\n" if scene_text else ""
    return (
        f"{ctx}"
        f"Câu hỏi: {question}\n"
        f"Hãy trả lời ngắn, nêu kết luận chính và (nếu có) căn cứ luật rút gọn."
    )

print("✅ VLM system prompt & helper loaded.")


✅ VLM system prompt & helper loaded.


In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, InstructBlipForConditionalGeneration

# ưu tiên decord, fallback OpenCV
_USE_DECORD = False
try:
    import decord
    decord.bridge.set_bridge("native")
    _USE_DECORD = True
    print("Sampler: decord")
except Exception:
    import cv2
    print("Sampler: OpenCV fallback")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VLM_MODEL = "Salesforce/instructblip-vicuna-7b"  # GPU yếu: "Salesforce/instructblip-flan-t5-xl"

_VLM = None
_VLM_PROC = None

def _vlm_init(model_name=VLM_MODEL):
    global _VLM, _VLM_PROC
    if _VLM is None:
        _VLM_PROC = AutoProcessor.from_pretrained(model_name)
        _VLM = InstructBlipForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=(torch.bfloat16 if DEVICE=="cuda" else torch.float32),
        ).to(DEVICE)
    return _VLM, _VLM_PROC

def _sample_frames(video_path: str, max_frames=8):
    if _USE_DECORD:
        import decord
        vr = decord.VideoReader(video_path)
        n = len(vr)
        if n == 0: raise ValueError("Video rỗng.")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        return [Image.fromarray(vr[i].asnumpy()) for i in idxs]
    else:
        import cv2
        cap = cv2.VideoCapture(video_path)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if n <= 0: raise ValueError("Không đọc được video.")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        imgs=[]
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, fr = cap.read()
            if ok:
                fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
                imgs.append(Image.fromarray(fr))
        cap.release()
        if not imgs: raise ValueError("Không lấy được khung hình.")
        return imgs

def vlm_answer(video_path: str, user_prompt: str, system_prompt: str = None,
               frames=8, max_new_tokens=180):
    model, proc = _vlm_init()
    images = _sample_frames(video_path, max_frames=frames)
    prompt = f"[System]\n{system_prompt}\n[User]\n{user_prompt}" if system_prompt else user_prompt
    inputs = proc(images=images, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return proc.batch_decode(out, skip_special_tokens=True)[0]

# Định nghĩa prompt hệ thống để tránh NameError
VLM_SYSTEM_PROMPT = (
  "Bạn là trợ lý lái xe RoadBuddy. Trả lời ngắn gọn, chắc chắn, ưu tiên an toàn "
  "và tuân thủ Luật GTĐB Việt Nam. Khi đèn/biển/vạch xung đột: đèn > biển > vạch. "
  "Nếu không đủ thông tin, hãy nói 'chưa đủ thông tin' và khuyên 'giảm tốc/giữ làn'."
)

print("✅ VLM inference ready: gọi vlm_answer(video_path, user_prompt, VLM_SYSTEM_PROMPT)")


Sampler: OpenCV fallback
✅ VLM inference ready: gọi vlm_answer(video_path, user_prompt, VLM_SYSTEM_PROMPT)


In [ ]:
# -------------------- IMPORTS & CONFIG --------------------
import os, re, json, shutil
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

# PdfReader + fallback
try:
    from pypdf import PdfReader
except Exception:
    from PyPDF2 import PdfReader  # fallback

# Dự án
RB_ROOT   = "roadbuddy"
KB_DIR    = f"{RB_ROOT}/kb"
PDF_DIR   = f"{KB_DIR}/pdfs"      # nơi chứa PDF luật
RAG_OUT   = f"{KB_DIR}/index"     # nơi lưu index/chunks/embeddings

# File PDF NGUỒN (đổi theo bạn)
LAW_NEW_SRC = "/content/luat/luatgiaothong.pdf"          # luật mới (bạn đã có)
LAW_OLD_SRC = "/content/luat/luatbosung.pdf"     # nếu có luật cũ; nếu chưa có, vẫn chạy được với 1 file

# Embedding model
EMB_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
CHUNKS_JSONL = f"{RAG_OUT}/chunks.jsonl"
FAISS_INDEX  = f"{RAG_OUT}/faiss.index"

# FAISS optional
try:
    import faiss  # type: ignore
    _FAISS_OK = True
except Exception:
    faiss = None
    _FAISS_OK = False
    print("ℹ️ FAISS không khả dụng → sẽ dùng numpy cosine fallback.")

# -------------------- UTILITIES --------------------
def _pdf_text(path):
    reader = PdfReader(path); txt=[]
    for p in reader.pages:
        t = (p.extract_text() or "").replace("\t"," ")
        t = re.sub(r"[ \u00A0]+"," ", t).strip()
        if t: txt.append(t)
    return "\n".join(txt)

def _chunk(text, max_tokens=220, overlap=40):
    ws = text.split(); out=[]; i=0
    while i < len(ws):
        j = min(i+max_tokens, len(ws))
        out.append(" ".join(ws[i:j]))
        i = j - overlap if j-overlap>i else j
    return out

# -------------------- COPY PDF VỀ THƯ MỤC DỰ ÁN --------------------
os.makedirs(PDF_DIR, exist_ok=True)
pdf_list = []
for src in [LAW_NEW_SRC, LAW_OLD_SRC]:
    if src and os.path.isfile(src):
        dst = f"{PDF_DIR}/{os.path.basename(src)}"
        if os.path.abspath(src) != os.path.abspath(dst):
            shutil.copy(src, dst)
        pdf_list.append(dst)
# Nếu không có luật cũ -> dùng 1 file
if not pdf_list:
    raise FileNotFoundError("Không tìm thấy PDF nào. Hãy chỉnh LAW_NEW_SRC/LAW_OLD_SRC.")
print("PDFs in project:", [Path(p).name for p in pdf_list])

# ƯU TIÊN NGUỒN (file mới ưu tiên điểm cao hơn một chút)
from pathlib import Path as _P
PRIORITY_WEIGHTS = {
    _P(pdf_list[0]).name: 0.12,   # file 1 (thường là luật mới)
}
if len(pdf_list) > 1:
    PRIORITY_WEIGHTS[_P(pdf_list[1]).name] = 0.00  # file 2 (luật cũ)

# -------------------- BUILD INDEX --------------------
def build_rag_index(pdf_paths):
    Path(RAG_OUT).mkdir(parents=True, exist_ok=True)
    corpus=[]
    for p in pdf_paths:
        if not Path(p).exists():
            print(f"⚠️ Không thấy PDF: {p}");
            continue
        raw = _pdf_text(p)
        parts = re.split(r"\n\s*(Điều\s+\d+[^:\n]*:?)", raw, flags=re.I)
        if len(parts)>1:
            for head, body in zip(parts[1::2], parts[2::2]):
                for ch in _chunk(head+"\n"+body):
                    corpus.append({"pdf": Path(p).name, "section": head.strip(), "text": ch})
        else:
            for ch in _chunk(raw):
                corpus.append({"pdf": Path(p).name, "section":"", "text": ch})
    if not corpus:
        raise RuntimeError("Không có dữ liệu để index.")

    model = SentenceTransformer(EMB_MODEL)
    emb = model.encode([c["text"] for c in corpus],
                       convert_to_numpy=True, normalize_embeddings=True,
                       show_progress_bar=True)

    # lưu corpus
    with open(CHUNKS_JSONL,"w",encoding="utf-8") as f:
        for c in corpus: f.write(json.dumps(c, ensure_ascii=False)+"\n")

    # lưu index / embeddings
    if _FAISS_OK:
        index = faiss.IndexFlatIP(emb.shape[1])  # cosine (đã normalize)
        index.add(emb)
        faiss.write_index(index, FAISS_INDEX)
        np.save(f"{RAG_OUT}/embeddings.npy", emb)  # optional
        print(f"✅ RAG indexed {len(corpus)} chunks → {FAISS_INDEX}")
    else:
        np.save(f"{RAG_OUT}/embeddings.npy", emb)
        print(f"✅ Saved embeddings (fallback) {emb.shape} → {RAG_OUT}/embeddings.npy")
    print(f"✅ Saved chunks → {CHUNKS_JSONL}")

# -------------------- SEARCH (ƯU TIÊN + ĐA DẠNG NGUỒN) --------------------
from collections import defaultdict

def _diversify_rank(items, k=5, per_source_cap=3):
    by_src = defaultdict(list)
    for it in items:
        by_src[it["pdf"]].append(it)
    # sort each source
    for s in by_src:
        by_src[s].sort(key=lambda x: x["score"], reverse=True)

    out, used = [], set()
    while len(out) < k:
        progressed = False
        for s in list(by_src.keys()):
            if len(out) >= k: break
            if not by_src[s]: continue
            # cap theo nguồn
            if sum(1 for x in out if x["pdf"]==s) >= per_source_cap:
                continue
            cand = by_src[s].pop(0)
            key = (cand["pdf"], cand["section"], cand["text"][:80])
            if key in used:
                continue
            out.append(cand); used.add(key); progressed = True
        if not progressed:
            break
    # nếu thiếu → bốc theo score
    if len(out) < k:
        pool = [it for lst in by_src.values() for it in lst]
        pool.sort(key=lambda x: x["score"], reverse=True)
        for cand in pool:
            if len(out) >= k: break
            key = (cand["pdf"], cand["section"], cand["text"][:80])
            if key in used: continue
            out.append(cand); used.add(key)
    return out[:k]

def rag_search_two_laws(query, k=5, diversify=True):
    chunks=[json.loads(x) for x in Path(CHUNKS_JSONL).read_text(encoding="utf-8").splitlines()]
    model = SentenceTransformer(EMB_MODEL)
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    if _FAISS_OK and Path(FAISS_INDEX).exists():
        K_RAW = max(k*6, 20)
        index = faiss.read_index(FAISS_INDEX)
        D, I = index.search(q, K_RAW)
        idxs, scores = I[0], D[0]
        items=[]
        for idx, sc in zip(idxs, scores):
            if idx < 0: continue
            c = chunks[int(idx)].copy()
            boost = float(PRIORITY_WEIGHTS.get(c["pdf"], 0.0))
            c["score"] = float(sc) + boost
            items.append(c)
        items.sort(key=lambda x: x["score"], reverse=True)
        return _diversify_rank(items, k=k) if diversify else items[:k]
    else:
        emb = np.load(f"{RAG_OUT}/embeddings.npy")
        raw_scores = (q @ emb.T)[0]  # cosine
        items=[]
        for i, sc in enumerate(raw_scores):
            c = chunks[i].copy()
            boost = float(PRIORITY_WEIGHTS.get(c["pdf"], 0.0))
            c["score"] = float(sc) + boost
            items.append(c)
        items.sort(key=lambda x: x["score"], reverse=True)
        return _diversify_rank(items, k=k) if diversify else items[:k]

# -------------------- BUILD & TEST --------------------
build_rag_index(pdf_list)

query = "tốc độ tối đa trong khu vực đô thị khi có biển tốc độ"
hits = rag_search_two_laws(query, k=4)
for i,h in enumerate(hits,1):
    print(f"\n[{i}] [{h['pdf']}] {h['section']}  score={h['score']:.3f}\n{h['text'][:300]}...")

PDFs in project: ['luatgiaothong.pdf', 'luatbosung.pdf']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

✅ RAG indexed 841 chunks → roadbuddy/kb/index/faiss.index
✅ Saved chunks → roadbuddy/kb/index/chunks.jsonl

[1] [luatgiaothong.pdf] Điều 15. Chuyển hướng xe  score=0.893
đoạn đường cong tầm nhìn bị che khuất, trên đường cao tốc, trong hầm đường bộ, trên đường một chiều, trừ khi có hiệu lệnh của người điều khiển giao thông hoặc chỉ dẫn của biển báo hiệu tạm thời....

[2] [luatbosung.pdf] Điều 35. Tác dụng của biển chỉ dẫn  score=0.785
Điều 35. Tác dụng của biển chỉ dẫn Các biển chỉ dẫn để chỉ dẫn hướng đi hoặc các điều cần thiết nhằm giúp người tham gia giao thông trong việc điều khiển phương tiện và hướng dẫn giao thông trên đường được thuận lợi, đảm bảo an toàn....

[3] [luatgiaothong.pdf] Điều 24. Giao thông tại đường ngang, cầu chung đường sắt  score=0.874
khỏi phạm vi an toàn đường sắt, người điều khiển phương tiện tham gia giao thông đường bộ và người có mặt phải ngay lập tức báo hiệu để dừng tàu, thực hiện các biện pháp bảo đảm an toàn....

[4] [luatbosung.pdf] Điều 32. Ý nghĩa s

In [ ]:
# ===================== Cell 2.9 — VLM Context Embedding (LITE, CLIP) =====================
# Nhẹ hơn: dùng CLIP ViT-B/32 để sinh embedding khung hình (không cần Q-Former).
# Yêu cầu: transformers, PIL; decord (nếu có) hoặc OpenCV fallback.

import json, math
from pathlib import Path
from typing import List
import numpy as np
import torch
from PIL import Image

# -------- Cấu hình --------
SCENE_DESC_PATH = "roadbuddy/runtime/scene_description.jsonl"   # input từ M2.1
EMB_DIR         = "roadbuddy/runtime/vlm_ctx"                    # nơi lưu nhúng
CLIP_MODEL      = "openai/clip-vit-base-patch32"                # rất nhẹ, phổ biến
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
FRAMES_PER_WIN  = 6                                             # số frame mẫu / cửa sổ
VIDEO_DIR       = "/content/video"                              # thư mục chứa .mp4
POOLING         = "mean"                                        # 'mean' (gợi ý)

# -------- Model/processor --------
from transformers import AutoProcessor, CLIPModel
_clip_proc  = AutoProcessor.from_pretrained(CLIP_MODEL)
_clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()

# -------- Sampler khung hình --------
try:
    import decord
    decord.bridge.set_bridge("native")
    _USE_DECORD = True
except Exception:
    import cv2
    _USE_DECORD = False

def _sample_frames_from_video(video_path: str, max_frames: int = 6) -> List[Image.Image]:
    if _USE_DECORD:
        vr = decord.VideoReader(video_path)
        n = len(vr)
        if n <= 0: raise ValueError(f"Không đọc được video: {video_path}")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        return [Image.fromarray(vr[i].asnumpy()) for i in idxs]
    else:
        import cv2
        cap = cv2.VideoCapture(video_path)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if n <= 0:
            cap.release()
            raise ValueError(f"Không đọc được video: {video_path}")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        imgs=[]
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, fr = cap.read()
            if ok:
                fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
                imgs.append(Image.fromarray(fr))
        cap.release()
        if not imgs: raise ValueError("Không lấy được khung hình.")
        return imgs

# -------- Hàm tạo embedding cho list ảnh (CLIP) --------
@torch.no_grad()
def clip_embed(images: List[Image.Image]) -> torch.Tensor:
    """
    Trả về vector (dim,) — mean-pool các frame sau khi lấy get_image_features.
    """
    inputs = _clip_proc(images=images, return_tensors="pt").to(DEVICE)
    feats  = _clip_model.get_image_features(pixel_values=inputs["pixel_values"])  # (bs, dim)
    feats  = torch.nn.functional.normalize(feats, dim=-1)
    if POOLING == "mean":
        v = feats.mean(dim=0)                # (dim,)
    else:
        v = feats[0]                         # frame đầu
    v = torch.nn.functional.normalize(v, dim=0)
    return v.detach().float().cpu()

# -------- Duyệt scene_description và lưu nhúng --------
Path(EMB_DIR).mkdir(parents=True, exist_ok=True)
meta_out = []
with open(SCENE_DESC_PATH, "r", encoding="utf-8") as f:
    windows = [json.loads(x) for x in f]

for w in windows:
    vid   = w["vid"]
    vpath = f"{VIDEO_DIR}/{vid}.mp4"   # nếu đuôi khác, đổi theo dữ liệu của bạn
    if not Path(vpath).exists():
        # thử các đuôi phổ biến khác nếu cần
        alt = None
        for ext in [".MP4",".mov",".MOV",".mkv",".MKV",".avi",".AVI"]:
            if Path(f"{VIDEO_DIR}/{vid}{ext}").exists():
                alt = f"{VIDEO_DIR}/{vid}{ext}"; break
        if not alt:
            # không tìm thấy video -> bỏ qua
            continue
        vpath = alt

    try:
        imgs = _sample_frames_from_video(vpath, max_frames=FRAMES_PER_WIN)
        vec  = clip_embed(imgs)                           # (dim,)
        npy_path = f"{EMB_DIR}/{vid}_{w['t0']:.3f}_{w['t1']:.3f}.npy"
        np.save(npy_path, vec.numpy())
        meta_out.append({"vid": vid, "t0": w["t0"], "t1": w["t1"], "embed_path": npy_path, "dim": int(vec.numel())})
    except Exception as e:
        print(f"[WARN] {vid} [{w['t0']}-{w['t1']}] lỗi nhúng: {e}")

with open(f"{EMB_DIR}/vlm_ctx_meta.jsonl", "w", encoding="utf-8") as f:
    for m in meta_out:
        f.write(json.dumps(m, ensure_ascii=False) + "\n")

print(f"✅ Saved {len(meta_out)} CLIP embeddings → {EMB_DIR}")
print("Ví dụ 1 dòng meta:", meta_out[0] if meta_out else "(trống)")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Saved 0 CLIP embeddings → roadbuddy/runtime/vlm_ctx
Ví dụ 1 dòng meta: (trống)


In [ ]:
# ============== M2.T — Temporal Facts per window ==============
import json, math
from collections import Counter, defaultdict
from pathlib import Path

SCENE_CACHE_PATH = "roadbuddy/runtime/scene_cache.jsonl"
SCENE_DESC_PATH  = "roadbuddy/runtime/scene_description.jsonl"
TEMP_FACTS_PATH  = "roadbuddy/runtime/temporal_facts.jsonl"
FPS, CONF_TH = 15, 0.35

def _iter_jsonl(p):
    with open(p,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: yield json.loads(line)

def _range_frames(t0, t1, fps=FPS):
    return range(int(math.floor(t0*fps)), int(math.ceil(t1*fps))+1)

def build_temporal_facts():
    if not Path(SCENE_CACHE_PATH).exists() or not Path(SCENE_DESC_PATH).exists():
        raise FileNotFoundError("Thiếu scene_cache hoặc scene_description.")
    by_vid = defaultdict(dict)
    for r in _iter_jsonl(SCENE_CACHE_PATH):
        by_vid[r["vid"]][int(r["frame"])] = r

    wrote=0
    with open(TEMP_FACTS_PATH,"w",encoding="utf-8") as out:
        for w in _iter_jsonl(SCENE_DESC_PATH):
            vid, t0, t1 = w["vid"], float(w["t0"]), float(w["t1"])
            frames = _range_frames(t0, t1, FPS)
            light_hist, sign_hist, lane_hist, ocr_hist = Counter(), Counter(), Counter(), Counter()
            consec = {"red":0,"yellow":0,"green":0}
            max_consec = {"red":0,"yellow":0,"green":0}

            for f in frames:
                r = by_vid.get(vid,{}).get(f)
                if not r:
                    for k in consec: consec[k]=0
                    continue
                # lights
                frame_has=set()
                for l in r.get("lights",[]):
                    if float(l.get("conf",0))>=CONF_TH:
                        cls=int(l.get("cls",-1))
                        name={0:"red",1:"yellow",2:"green",3:"green_left",4:"green_straight"}.get(cls,f"light_{cls}")
                        light_hist[name]+=1
                        if name in ("red","yellow","green"): frame_has.add(name)
                for k in consec:
                    if k in frame_has:
                        consec[k]+=1; max_consec[k]=max(max_consec[k],consec[k])
                    else:
                        consec[k]=0
                # signs / lanes / ocr
                for s in r.get("signs",[]):
                    if float(s.get("conf",0))>=CONF_TH:
                        sign_hist[str(s.get("name") or f"sign_{s.get('cls',-1)}")]+=1
                for ln in r.get("lanes",[]):
                    t=ln.get("type");
                    if t: lane_hist[t]+=1
                for o in r.get("ocr",[]):
                    txt=str(o.get("text","")).strip()
                    if txt: ocr_hist[txt]+=1

            dur=lambda c: round(c/float(FPS),3)
            facts={
                "vid":vid,"t0":t0,"t1":t1,
                "light_dominant": (light_hist.most_common(1)[0][0] if light_hist else "unknown"),
                "red_stable_sec": dur(max_consec["red"]),
                "green_stable_sec": dur(max_consec["green"]),
                "yellow_stable_sec": dur(max_consec["yellow"]),
                "sign_top":[k for k,_ in sign_hist.most_common(5)],
                "lane_top":[k for k,_ in lane_hist.most_common(4)],
                "ocr_top":[k for k,_ in ocr_hist.most_common(6)],
            }
            out.write(json.dumps(facts,ensure_ascii=False)+"\n"); wrote+=1
    print(f"✅ Temporal facts: {wrote} đoạn → {TEMP_FACTS_PATH}")

# chạy 1 lần sau khi có scene_description.jsonl
build_temporal_facts()


✅ Temporal facts: 151 đoạn → roadbuddy/runtime/temporal_facts.jsonl


Module3


In [ ]:
# ==================== M3.0 — ONE-TIME INSTALL (Module 3 only) ====================
!pip -q install --upgrade transformers accelerate
!pip -q install --upgrade opencv-python easyocr
# Phi-3 mini là mô hình text LLM → không cần timm/pillow/decord.
# RAG chỉ cần sentence-transformers/faiss *nếu* bạn chạy lại RAG; còn ở M3 ta chỉ "gọi" hàm RAG có sẵn.

import importlib
def _ok(pkg, alias=None):
    try:
        importlib.import_module(alias or pkg)
        print(f"✅ {pkg} OK")
    except Exception as e:
        print(f"⚠️ {pkg} không sẵn sàng:", e)

_ok("transformers")
_ok("accelerate")
_ok("opencv-python","cv2")
_ok("easyocr")
print("Gợi ý: Nếu muốn gọi RAG ở M3, hãy đảm bảo đã chạy Module 2 để có rag_search_two_laws/rag_search_best.")


✅ transformers OK
✅ accelerate OK
✅ opencv-python OK
✅ easyocr OK
Gợi ý: Nếu muốn gọi RAG ở M3, hãy đảm bảo đã chạy Module 2 để có rag_search_two_laws/rag_search_best.


In [ ]:
# ====================== M3.1 — CONFIG & PATHS ======================
from pathlib import Path

QUESTIONS_JSON = "/content/data/public_test.json"
META_JSON      = "/content/runs_meta/metadata.json"
NORM_JSON      = "/content/data/label.json"
VIDEO_ROOT     = "/content/drive/MyDrive/videos"

# OUTPUT
OUT_SHORT = "roadbuddy/runtime/answers_short.csv"
OUT_FULL  = "roadbuddy/runtime/answers_full.csv"

# SCENE / TEMPORAL / RAG
SCENE_DESC_PATH = "roadbuddy/runtime/scene_description.jsonl"
TEMPORAL_PATH   = "roadbuddy/runtime/temporal_windows.jsonl"
FAISS_INDEX_PATH = "roadbuddy/kb/index/faiss.index"
CHUNKS_JSONL     = "roadbuddy/kb/index/chunks.jsonl"

RAG_TOPK = 3
OCR_MAX_FRAMES = 6

PHI_MODEL = "microsoft/Phi-3-mini-4k-instruct"
USE_PHI   = True

# Ensure output folder exists
Path(Path(OUT_SHORT).parent).mkdir(parents=True, exist_ok=True)
Path(Path(OUT_FULL ).parent).mkdir(parents=True, exist_ok=True)

print("OK M3.1")


OK M3.1


In [ ]:
# ====================== M3.2 — IMPORTS ======================
import os, re, json, time, csv, statistics
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import cv2
import easyocr
import faiss

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("OK M3.2")


OK M3.2


In [ ]:
# === M3.PATCH — Loader file câu hỏi (JSON / JSONL / CSV) ===
import json, csv, re
from pathlib import Path
from typing import List, Dict, Any, Optional

def _clean_choices(choices) -> List[str]:
    """Chuẩn hóa mảng đáp án: bỏ tiền tố 'A. ', 'B) ', '(C)', '[D]:' ở đầu; trim."""
    out = []
    if isinstance(choices, dict):  # {"A":"...","B":"..."}
        ordered = []
        for k in ["A","B","C","D","a","b","c","d"]:
            if k in choices: ordered.append(choices[k])
        choices = ordered if ordered else list(choices.values())

    if isinstance(choices, str):
        # hỗ trợ chuỗi 'A||B||C||D'
        choices = [x.strip() for x in choices.split("||") if x.strip()]

    if not isinstance(choices, list):
        choices = []

    for ch in choices:
        s = str(ch)
        s = re.sub(r'^\s*[\(\[\{]?[A-Da-d][\)\]\}\.:]\s*', '', s).strip()
        out.append(s)
    return out

def _normalize_row(row: Any) -> Optional[Dict[str, Any]]:
    """Đưa 1 bản ghi về schema chuẩn {id, question, options(list), video}."""
    if isinstance(row, str):
        try:
            row = json.loads(row)
        except Exception:
            return None
    if not isinstance(row, dict):
        return None

    qid   = row.get("id") or row.get("ID") or row.get("qid")
    qtext = row.get("question") or row.get("q") or row.get("prompt")
    opts  = row.get("choices") or row.get("options") or row.get("opts")
    vpath = row.get("video_path") or row.get("video") or row.get("path") or row.get("videoFile")

    opts = _clean_choices(opts)
    if not (qid and qtext and vpath):
        return None
    return {"id": qid, "question": qtext, "options": opts, "video": vpath}

def load_questions(path: str) -> List[Dict[str, Any]]:
    """Đọc file câu hỏi:
       - JSON { "data": [...] } (schema của bạn) hoặc list thuần
       - JSONL (mỗi dòng 1 JSON)
       - CSV (có header: id, question/choices/options/video_path/video)
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Không thấy file câu hỏi: {path}")
    text = p.read_text(encoding="utf-8").strip()

    recs: List[Dict[str,Any]] = []

    # 1) Thử JSON
    try:
        obj = json.loads(text)
        rows = obj.get("data", obj if isinstance(obj, list) else [])
        if not isinstance(rows, list):
            rows = [rows]
        for r in rows:
            nr = _normalize_row(r)
            if nr: recs.append(nr)
    except Exception:
        # 2) JSONL
        parsed = 0
        for ln in text.splitlines():
            ln = ln.strip()
            if not ln: continue
            try:
                r = json.loads(ln)
                nr = _normalize_row(r)
                if nr:
                    recs.append(nr); parsed += 1
            except Exception:
                pass
        # 3) CSV nếu JSON/JSONL đều không parse được
        if parsed == 0:
            with open(p, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    nr = _normalize_row(row)
                    if nr: recs.append(nr)

    # Lọc và báo cáo
    recs = [r for r in recs if r["id"] and r["question"] and r["video"]]
    print(f"✅ load_questions: {len(recs)} câu từ {path}")
    if recs:
        demo = {k: recs[0][k] for k in ["id","question","options","video"]}
        print("Ví dụ:", demo)
    else:
        print("⚠️ Không có bản ghi hợp lệ. Kiểm tra khóa: id / question / choices / video_path.")
    return recs


In [ ]:
# ====================== M3.3 — LOADERS (FIXED) ======================
from pathlib import Path
import json

def load_questions(path):
    txt = Path(path).read_text(encoding="utf-8")

    # parse file
    data = json.loads(txt)

    out = []
    for r in data:

        # ==== FIX QUAN TRỌNG ====
        # Nếu phần tử là STRING → parse lại thành dict
        if isinstance(r, str):
            try:
                r = json.loads(r)
            except Exception as e:
                print("⚠️ Không parse được dòng JSON:", r)
                continue
        # ==========================

        out.append({
            "id": str(r.get("id","")),
            "question": r.get("question") or r.get("q",""),
            "options": r.get("options") or r.get("opts") or [],
            "video": r.get("video") or r.get("video_path") or ""
        })
    return out


def _norm_key(p: str) -> str:
    return Path(p).name.strip()


def load_meta(path: str):
    mp = {}
    if not Path(path).exists(): return mp
    raw = json.loads(Path(path).read_text())
    for it in raw.get("items", []):
        k = _norm_key(it.get("video_path",""))
        objs=[]
        for kf in it.get("keyframes", []):
            for o in kf.get("objects", []):
                objs.append({"name":o["name"],"cls":o["cls"],"conf":o["conf"]})
        mp[k]=objs
    return mp


def load_norm(path: str):
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    return {k.lower():v for k,v in raw.items()}


def load_scene_desc_map(path=SCENE_DESC_PATH):
    mp={}
    if not Path(path).exists(): return mp
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            r=json.loads(ln)
            k=_norm_key(r["vid"])
            mp.setdefault(k,[]).append(r)
    return mp


def load_temporal_map(path=TEMPORAL_PATH):
    mp={}
    if not Path(path).exists(): return mp
    with open(path,"r",encoding="utf-8") as f:
        for ln in f:
            r=json.loads(ln)
            k=_norm_key(r["vid"])
            mp.setdefault(k,[]).append(r)
    return mp

print("OK M3.3 FIXED")


OK M3.3


In [ ]:
# ====================== M3.4 — NORMALIZE META LABELS ======================

from collections import Counter

def normalize_names(raw_objs, norm_map, min_conf=0.15):
    out=[]
    for o in raw_objs:
        if float(o["conf"]) < min_conf: continue
        base = o["name"].strip()
        vn   = norm_map.get(base.lower(), base)
        out.append(vn)
    return out[:12]

print("OK M3.4")


OK M3.4


In [ ]:
# ====================== M3.5 — OCR ======================
ocr_reader = easyocr.Reader(['vi','en'], gpu=torch.cuda.is_available())

def sample_frames(video_path, max_frames=OCR_MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return []
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(n-1,1), max_frames, dtype=int)
    frames=[]
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, fr = cap.read()
        if ok: frames.append(fr)
    cap.release()
    return frames

def ocr_video_texts(video_path):
    if not Path(video_path).exists():
        alt = str(Path(VIDEO_ROOT)/Path(video_path).name)
        if Path(alt).exists(): video_path=alt
        else: return []

    texts=[]
    for fr in sample_frames(video_path):
        rgb = fr[:,:,::-1]
        res = ocr_reader.readtext(rgb, detail=0)
        for t in res:
            t = re.sub(r"\s+"," ",t).strip()
            if len(t)>=2 and t not in texts:
                texts.append(t)
    return texts[:20]

print("OK M3.5")


OK M3.5


In [ ]:
# ====================== M3.6 — RAG FAISS ======================

# load FAISS
rag_index = faiss.read_index(FAISS_INDEX_PATH) if Path(FAISS_INDEX_PATH).exists() else None

# load chunks
rag_chunks=[]
if Path(CHUNKS_JSONL).exists():
    with open(CHUNKS_JSONL,"r",encoding="utf-8") as f:
        for ln in f:
            rag_chunks.append(json.loads(ln))

def rag_fetch(query: str, k=RAG_TOPK):
    if rag_index is None or not rag_chunks:
        return []
    # query vector tạm
    qvec = np.random.rand(1, rag_index.d).astype("float32")
    D,I = rag_index.search(qvec, k)
    return [rag_chunks[i]["text"] for i in I[0] if i < len(rag_chunks)]

print("OK M3.6")


OK M3.6


In [ ]:
# ====================== M3.7 — SCENE & TEMPORAL ======================

def scene_text_for_video(vkey, scene_map, limit=3):
    snaps = scene_map.get(vkey, [])
    snaps = sorted(snaps, key=lambda x:x.get("t0",0))[:limit]
    return " ".join(f"[{s['t0']}-{s['t1']}s] {s['description']}" for s in snaps)

def rule_engine_temporal(vkey, temporal_map):
    ws = temporal_map.get(vkey, [])
    if not ws: return []
    w = sorted(ws, key=lambda x:x.get("t0",0))[0]
    adv=[]
    if w.get("red_stable_sec",0)>=2: adv.append("Đèn đỏ ổn định ≥2s: phải dừng trước vạch.")
    if w.get("green_stable_sec",0)>=2: adv.append("Đèn xanh ổn định: được đi nếu an toàn.")
    if w.get("yellow_stable_sec",0)>=1: adv.append("Đèn vàng: chuẩn bị dừng.")
    # speed from OCR
    txt = " ".join(w.get("ocr_top", []))
    m = re.findall(r"\b(\d{2,3})\b", txt)
    sp = [int(x) for x in m if 15<=int(x)<=120]
    if sp: adv.append(f"Tốc độ tối đa: {max(sp)} km/h.")
    return adv

print("OK M3.7")


OK M3.7


In [ ]:
# ====================== M3.8 — PHI-3 MINI ======================

_USE_PHI_OK = False
if USE_PHI:
    try:
        tok = AutoTokenizer.from_pretrained(PHI_MODEL, trust_remote_code=True)
        phi = AutoModelForCausalLM.from_pretrained(
            PHI_MODEL,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True
        )
        _USE_PHI_OK=True
        print("Phi-3 mini loaded OK.")
    except Exception as e:
        print("Phi load fail:",e)

def build_prompt(q, options, scene, labels, ocr, laws, temp):
    opt_txt = "\n".join(f"{chr(65+i)}. {x}" for i,x in enumerate(options))
    return (
        "Bạn là trợ lý giao thông. Chỉ trả lời A/B/C/D ở dòng đầu.\n"
        f"Câu hỏi: {q}\n{opt_txt}\n\n"
        f"Cảnh: {scene}\n"
        f"Nhãn: {', '.join(labels)}\n"
        f"OCR: {', '.join(ocr)}\n"
        f"Luật: {', '.join(laws)}\n"
        f"Thời gian: {', '.join(temp)}\n\nĐÁP ÁN:"
    )

def infer_phi(q, options, scene, labels, ocr, laws, temp):
    if not _USE_PHI_OK:
        return "A","fallback"
    prompt = build_prompt(q, options, scene, labels, ocr, laws, temp)
    inp = tok(prompt, return_tensors="pt").to(phi.device)
    with torch.no_grad():
        out = phi.generate(**inp, max_new_tokens=120, do_sample=False)
    txt = tok.decode(out[0], skip_special_tokens=True)
    line0 = txt.strip().split("\n",1)[0]
    m = re.search(r"\b([ABCD])\b", line0)
    ans = m.group(1) if m else "A"
    expl = txt[len(line0):].strip()
    return ans, expl[:300]

print("OK M3.8")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Phi-3 mini loaded OK.
OK M3.8


In [ ]:
# ====================== M3.9 — answer_one ======================

def answer_one(r, meta_map, norm_map, scene_map, temporal_map):
    qid   = r["id"]
    qtext = r["question"]
    opts  = r.get("options",[])
    vpath = r["video"]
    vkey  = _norm_key(vpath)

    # meta
    raw_objs = meta_map.get(vkey, [])
    norm_labels = normalize_names(raw_objs, norm_map)

    # ocr
    video_abs = str(Path(VIDEO_ROOT)/vkey)
    ocr_list = ocr_video_texts(video_abs)

    # scene + temporal
    scene_text = scene_text_for_video(vkey, scene_map) if vkey in scene_map else ""
    temporal_advice = rule_engine_temporal(vkey, temporal_map)

    # RAG
    law_snips = rag_fetch(qtext)

    # Phi
    ansL, expl = infer_phi(qtext, opts, scene_text, norm_labels, ocr_list, law_snips, temporal_advice)

    # map answer
    idx = ord(ansL)-65
    ansT = opts[idx] if 0<=idx<len(opts) else ""

    return {
        "id":qid,
        "video_path":vpath,
        "question":qtext,
        "options":opts,
        "answer_letter":ansL,
        "answer_text":ansT,
        "explain":expl,
        "scene_text":scene_text,
        "temporal_advice":" | ".join(temporal_advice),
        "norm_labels":"; ".join(norm_labels),
        "ocr_texts":" | ".join(ocr_list),
        "law_refs":" | ".join(law_snips),
    }

print("OK M3.9")


OK M3.9


In [ ]:
# ====================== M3.10 — STREAMING RUNNER ======================

def run_module3_streaming(
    questions_json=QUESTIONS_JSON,
    meta_json=META_JSON,
    norm_json=NORM_JSON,
    out_short=OUT_SHORT,
    out_full=OUT_FULL):

    # load
    qrows = load_questions(questions_json)
    meta_map = load_meta(meta_json)
    norm_map = load_norm(norm_json)
    scene_map = load_scene_desc_map()
    temporal_map = load_temporal_map()

    # prepare output
    short_p = Path(out_short)
    full_p  = Path(out_full)
    short_p.parent.mkdir(parents=True, exist_ok=True)
    full_p.parent.mkdir(parents=True, exist_ok=True)

    full_header=[
        "id","video_path","question","options","answer_letter","answer_text",
        "explain","scene_text","temporal_advice","norm_labels","ocr_texts","law_refs"
    ]

    with open(out_full,"w",encoding="utf-8",newline="") as f:
        csv.DictWriter(f, fieldnames=full_header).writeheader()

    print(f"[RUN] {len(qrows)} câu…")

    for idx,r in enumerate(qrows,1):
        try:
            out = answer_one(r, meta_map, norm_map, scene_map, temporal_map)

            # short
            with open(out_short,"a",encoding="utf-8",newline="") as f:
                csv.writer(f).writerow([f"câu{out['id']}:{out['answer_letter']}"])

            # full
            row = dict(out); row["options"]=" || ".join(row["options"])
            with open(out_full,"a",encoding="utf-8",newline="") as f:
                csv.DictWriter(f,fieldnames=full_header).writerow(row)

        except Exception as e:
            print("ERR:", e)

        if idx%5==0:
            print(f"  {idx}/{len(qrows)} done…")

    print("🎉 XONG! →", out_short, "&", out_full)
    return out_short, out_full

print("OK M3.10")


OK M3.10


In [ ]:
#run_module3_export(BATCH=25)

run_module3_streaming()


AttributeError: 'str' object has no attribute 'get'